# CA WUI Pre-Fire Building Footprints — Batch Compiler
## Structure Separation Distance Density (SSDD) Research Pipeline

**Purpose:** For each CA WUI fire in the inventory, compile the best available
pre-fire building footprint dataset by combining:

1. **Historical OSM** via ohsome API — exact snapshot at fire date − 1 day
2. **ML buildings** via Overture Maps S3 — non-OSM gap-fill
3. **DINS** via live CAL FIRE ArcGIS REST API — field-verified structure inventory  
   used to correct post-fire survivorship bias in the ML dataset

**Data cutoff:** Fires from Camp Fire (2018-11-08) onwards — reliable pre-fire  
footprint data exists back to 2017-01-01 for CA WUI communities.

**Output per fire:**
- `_prefire_buildings.gpkg` — OSM + ML merged, clipped, with spatial metrics
- `_unified_structures.gpkg` — DINS-centric hybrid (real polygons + circular estimates)
- `_source_map.png`, `_spacing.png` — QA visualisations
- `_summary.json` — run metadata

**Run options:**  
- Sections 11–12 below: run a single fire interactively  
- Section 13: batch all fires in one loop


---
## 0. Setup

```bash
pip install requests geopandas pyarrow s3fs shapely scipy matplotlib tqdm
```

In [ ]:
import json, math, time, warnings
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely import wkb
from shapely.geometry import Point, shape
from shapely.strtree import STRtree
from scipy.spatial import cKDTree

try:
    from tqdm.auto import tqdm
    TQDM = True
except ImportError:
    def tqdm(x, **kw): return x
    TQDM = False

warnings.filterwarnings('ignore')
print('Imports OK')


---
## 1. Configuration

Edit the paths and processing parameters here.  All other cells read from this cell.

### DINS caching
The live DINS API results are cached as parquet files in `DINS_CACHE_DIR`.  
On subsequent runs the cache is used instead of re-fetching, unless you  
delete the file or set `DINS_FORCE_REFRESH = True`.


In [ ]:
# ── Output directories ────────────────────────────────────────────────
BASE_DIR       = Path('.')                  # same folder as this notebook
OUTPUT_ROOT    = BASE_DIR / 'fires'         # one sub-folder per fire
DINS_CACHE_DIR = BASE_DIR / 'DINS'         # cached DINS parquet files

OUTPUT_ROOT.mkdir(exist_ok=True)
DINS_CACHE_DIR.mkdir(exist_ok=True)

# ── DINS API ──────────────────────────────────────────────────────────
DINS_MASTER_URL = (
    "https://services1.arcgis.com/jUJYIo9tSA7EHvfZ/arcgis/rest/services/"
    "POSTFIRE_MASTER_DATA_SHARE/FeatureServer/0/query"
)
DINS_PAGE_SIZE    = 2000
DINS_FORCE_REFRESH = False

# ── Processing parameters ─────────────────────────────────────────────
IOU_THRESHOLD  = 0.30
MATCH_M        = 25
WALL_SEARCH_R  = 60
MIN_AREA_M2    = 15

DINS_EXCLUDE = {'Other Minor Structure', 'Infrastructure', 'Agriculture'}

# ── External API URLs ─────────────────────────────────────────────────
CALFIRE_URL = (
    "https://egis.fire.ca.gov/arcgis/rest/services/"
    "FRAP/FirePerimeters_FS/FeatureServer/0/query"
)
CALFIRE_URL_FALLBACK = (
    "https://services1.arcgis.com/jUJYIo9tSA7EHvfZ/arcgis/rest/services/"
    "California_Historic_Fire_Perimeters/FeatureServer/0/query"
)
OHSOME_URL = "https://api.ohsome.org/v1/elements/geometry"

# ── Fire Inventory ────────────────────────────────────────────────────
# Fields:
#   name       — slug for output filenames
#   date       — ignition date; OSM queried at date − 1 day
#   where      — CAL FIRE FRAP SQL filter (perimeter layer)
#   dins_name  — INCIDENTNAME in DINS master table (None = skip)
#   dins_where — (optional) full WHERE clause overriding default LIKE construction;
#                use for ambiguous names to avoid contamination
#
# Fires NOT included:
#   Beckwourth (2021-07-02) — no FRAP perimeter; USFS-managed
#   Mountain   (2021-08-17) — DINS 'MOUNTAIN' matches unrelated fires statewide
FIRE_INVENTORY = [
    # ── 2018 ──────────────────────────────────────────────────────────
    dict(name='Camp',                  date='2018-11-08',
         where="FIRE_NAME='CAMP' AND YEAR_=2018",
         dins_name='Camp',
         dins_where="INCIDENTNAME = 'Camp'"),   # avoids 'Happy Camp Complex'
    dict(name='Woolsey',               date='2018-11-08',
         where="FIRE_NAME='WOOLSEY' AND YEAR_=2018",
         dins_name='WOOLSEY'),
    # ── 2019 ──────────────────────────────────────────────────────────
    dict(name='Kincade',               date='2019-10-23',
         where="FIRE_NAME='KINCADE' AND YEAR_=2019",
         dins_name='KINCADE'),
    # ── 2020 ──────────────────────────────────────────────────────────
    dict(name='CZU_Lightning_Complex', date='2020-08-16',
         where="FIRE_NAME='CZU LIGHTNING COMPLEX' AND YEAR_=2020",
         dins_name='CZU'),
    dict(name='LNU_Lightning_Complex', date='2020-08-17',
         # Hennessey = dominant fire (192k of 363k total acres)
         where="FIRE_NAME='HENNESSEY' AND YEAR_=2020",
         dins_name='LNU'),
    dict(name='North_Complex',         date='2020-08-17',
         where="FIRE_NAME='NORTH COMPLEX' AND YEAR_=2020",
         dins_name='NORTH COMPLEX'),
    dict(name='SCU_Lightning_Complex', date='2020-08-18',
         where="FIRE_NAME='SCU LIGHTNING COMPLEX' AND YEAR_=2020",
         dins_name='SCU'),
    dict(name='Castle',                date='2020-08-19',
         where="FIRE_NAME='CASTLE' AND YEAR_=2020",    # FRAP name; DINS = SQF Complex
         dins_name='SQF Complex',
         dins_where="INCIDENTNAME = 'SQF Complex'"),
    dict(name='Creek',                 date='2020-09-04',
         where="FIRE_NAME='CREEK' AND YEAR_=2020",
         dins_name='CREEK'),
    dict(name='Slater',                date='2020-09-08',
         where="FIRE_NAME='SLATER' AND YEAR_=2020",
         dins_name='Slater'),
    dict(name='Glass',                 date='2020-09-27',
         where="FIRE_NAME='GLASS' AND YEAR_=2020",
         dins_name='GLASS'),
    dict(name='Zogg',                  date='2020-09-27',
         where="FIRE_NAME='ZOGG' AND YEAR_=2020",
         dins_name='ZOGG'),
    # ── 2021 ──────────────────────────────────────────────────────────
    dict(name='River',                 date='2021-07-11',
         # 4 fires named RIVER in FRAP 2021; GIS_ACRES isolates the Nevada County
         # fire (~9,655 ac) — more portable than DATE syntax across FRAP servers
         where="FIRE_NAME='RIVER' AND YEAR_=2021 AND GIS_ACRES > 5000",
         dins_name='River',
         dins_where="INCIDENTNAME = 'River'"),  # avoids 'Driver', 'Owens River'
    dict(name='Dixie',                 date='2021-07-13',
         where="FIRE_NAME='DIXIE' AND YEAR_=2021",
         dins_name='DIXIE'),
    dict(name='Caldor',                date='2021-08-14',
         where="FIRE_NAME='CALDOR' AND YEAR_=2021",
         dins_name='CALDOR'),
    # ── 2022 ──────────────────────────────────────────────────────────
    dict(name='Oak',                   date='2022-07-22',
         where="FIRE_NAME='OAK' AND YEAR_=2022",
         dins_name='OAK'),
    dict(name='McKinney',              date='2022-07-26',
         # GIS_ACRES isolates the Siskiyou County fire (~55,000 ac) from any
         # smaller fires with the same name in FRAP
         where="FIRE_NAME='MCKINNEY' AND YEAR_=2022 AND GIS_ACRES > 10000",
         dins_name='MCKINNEY'),
    # ── 2024 ──────────────────────────────────────────────────────────
    dict(name='Borel',                 date='2024-07-24',
         where="FIRE_NAME='BOREL' AND YEAR_=2024",
         dins_name='Borel'),
    dict(name='Park',                  date='2024-07-24',
         where="FIRE_NAME='PARK' AND YEAR_=2024",
         dins_name='PARK'),
    dict(name='Airport',               date='2024-09-09',
         where="FIRE_NAME='AIRPORT' AND YEAR_=2024",
         dins_name='AIRPORT'),
    # ── 2025 ──────────────────────────────────────────────────────────
    dict(name='Palisades',             date='2025-01-07',
         where="FIRE_NAME='PALISADES' AND YEAR_=2025",
         dins_name='PALISADES'),
    dict(name='Eaton',                 date='2025-01-07',
         where="FIRE_NAME='EATON' AND YEAR_=2025",
         dins_name='EATON'),
]

print(f"Configured {len(FIRE_INVENTORY)} fires")
print(f"Output root : {OUTPUT_ROOT.resolve()}")
print(f"DINS cache  : {DINS_CACHE_DIR.resolve()}")

---
## 2. DINS Data — Live CAL FIRE API

CAL FIRE publishes the DINS (Damage INSpection) dataset as a live ArcGIS  
FeatureService at:

```
POSTFIRE_MASTER_DATA_SHARE/FeatureServer/0
```

This table is the consolidated master record covering all CA fires 2013–present  
and is updated continuously as new inspections are completed — including ongoing  
events like the 2025 LA fires.

### Query strategy
- Filter by `INCIDENT_NAME LIKE '%<fire>%'` on the master table  
- Paginate with `resultOffset` (max 2,000 records per request)  
- Cache results as parquet so subsequent runs skip the fetch  
- Fallback: load any pre-existing local `.parquet` in `DINS_CACHE_DIR`

### Field mapping
The REST API may return fields with slightly different names depending on the  
service version.  The `_map_dins_fields()` helper normalises them to the  
standard schema used throughout this notebook:  
`STRUCTUREC`, `STRUCTURET`, `DAMAGE`, `LATITUDE`, `LONGITUDE`


In [ ]:
def _map_dins_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise DINS field names from the ArcGIS API to the standard schema.
    Handles common variants returned by different service versions.
    """
    rename = {}
    col_upper = {c.upper(): c for c in df.columns}

    # Structure category
    for candidate in ['STRUCTUREC', 'STRUCTURE_CATEGORY', 'STRUCTURECATEGORY',
                       'STRUCT_CAT', 'CATEGORY']:
        if candidate in col_upper:
            rename[col_upper[candidate]] = 'STRUCTUREC'
            break

    # Structure type
    for candidate in ['STRUCTURET', 'STRUCTURE_TYPE', 'STRUCTURETYPE', 'STRUCT_TYPE']:
        if candidate in col_upper:
            rename[col_upper[candidate]] = 'STRUCTURET'
            break

    # Damage
    for candidate in ['DAMAGE', 'DAMAGE_RATING', 'DAMAGE_LEVEL', 'DAMAGE_CAT']:
        if candidate in col_upper:
            rename[col_upper[candidate]] = 'DAMAGE'
            break

    # Coordinates
    for candidate in ['LATITUDE', 'LAT', 'Y']:
        if candidate in col_upper:
            rename[col_upper[candidate]] = 'LATITUDE'
            break
    for candidate in ['LONGITUDE', 'LON', 'LONG', 'X']:
        if candidate in col_upper:
            rename[col_upper[candidate]] = 'LONGITUDE'
            break

    return df.rename(columns=rename)


def fetch_dins_for_fire(
    dins_name: str,
    cache_dir: Path,
    master_url: str = DINS_MASTER_URL,
    page_size:  int = DINS_PAGE_SIZE,
    force:      bool = DINS_FORCE_REFRESH,
    where_override: str | None = None,
) -> gpd.GeoDataFrame | None:
    """
    Fetch DINS records for one fire from the CAL FIRE master table.

    Steps:
      1. Return cached parquet if it exists and force=False
      2. Query POSTFIRE_MASTER_DATA_SHARE with INCIDENTNAME filter
      3. Paginate until all records retrieved
      4. Normalise field names; rebuild WGS84 point geometry
      5. Cache result as parquet; return GeoDataFrame
      6. On any API failure, fall back to local parquet if present

    Parameters
    ----------
    dins_name      : INCIDENTNAME value in the DINS master table (e.g. 'DIXIE').
                     Mixed case is fine — LIKE is case-insensitive on this service.
                     Do NOT wrap in UPPER(); that function is unsupported and returns 400.
                     Also used as the cache filename stem (DINS_<dins_name>.parquet).
    cache_dir      : directory to read/write cached parquet files
    master_url     : CAL FIRE POSTFIRE_MASTER_DATA_SHARE FeatureServer query URL
    page_size      : records per API request (ArcGIS max = 2000)
    force          : if True, bypass cache and re-fetch from API
    where_override : if provided, used as the full SQL WHERE clause instead of
                     the default "INCIDENTNAME LIKE '%{dins_name}%'" construction.
                     Use this for fires whose INCIDENTNAME requires an exact match
                     (e.g. "INCIDENTNAME = 'Camp'" to avoid 'Happy Camp Complex').

    Returns
    -------
    GeoDataFrame with primary structures (DINS_EXCLUDE classes removed),
    WGS84 point geometry, or None if no data found.
    """
    if dins_name is None:
        return None

    cache_path = cache_dir / f"DINS_{dins_name.replace(' ', '_')}.parquet"

    # ── Return cached result ──────────────────────────────────────────
    if cache_path.exists() and not force:
        print(f"  DINS: loading cached {cache_path.name}")
        gdf = gpd.read_parquet(cache_path)
        print(f"  DINS: {len(gdf):,} primary structures (from cache)")
        return gdf

    # ── Build WHERE clause ────────────────────────────────────────────
    where_clause = (where_override if where_override
                    else f"INCIDENTNAME LIKE '%{dins_name}%'")

    # ── Fetch from API with pagination ────────────────────────────────
    all_features = []
    offset = 0
    print(f"  DINS: fetching '{dins_name}' from CAL FIRE API…")
    print(f"  DINS: WHERE {where_clause}")

    try:
        while True:
            params = {
                # ArcGIS LIKE is case-insensitive — UPPER() causes a 400 error
                'where':            where_clause,
                # '*' avoids 400 errors from requesting non-existent alt-name
                # fields; _map_dins_fields() normalises whatever comes back
                'outFields':        '*',
                'returnGeometry':   'false',   # we rebuild from LAT/LON
                'resultOffset':     offset,
                'resultRecordCount': page_size,
                'f':                'json',
            }
            resp = requests.get(master_url, params=params, timeout=60)
            resp.raise_for_status()
            data = resp.json()

            # Detect API-level errors (returned as JSON, not HTTP error)
            if 'error' in data:
                raise RuntimeError(f"DINS API error: {data['error']}")

            features = data.get('features', [])
            all_features.extend(features)
            print(f"    page offset={offset}: {len(features)} records"
                  f"  (total so far: {len(all_features)})")

            # Stop when fewer records returned than requested
            if len(features) < page_size:
                break
            offset += page_size

        if not all_features:
            print(f"  DINS: no records found for '{dins_name}'")
            # Fall through to local fallback below
            raise ValueError("No records returned")

        # ── Parse features into DataFrame ─────────────────────────────
        rows = [f['attributes'] for f in all_features]
        df   = pd.DataFrame(rows)
        df   = _map_dins_fields(df)

        # Validate required columns
        for req in ['LATITUDE', 'LONGITUDE']:
            if req not in df.columns:
                raise ValueError(f"Required field '{req}' missing from API response. "
                                 f"Available: {list(df.columns)}")

        # Drop records with invalid coordinates
        df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])
        df = df[(df['LATITUDE'] != 0) & (df['LONGITUDE'] != 0)]

        # Exclude minor structure classes
        if 'STRUCTUREC' in df.columns:
            df = df[~df['STRUCTUREC'].isin(DINS_EXCLUDE)].copy()

        # Build WGS84 point geometry
        df['geometry'] = [Point(r.LONGITUDE, r.LATITUDE) for r in df.itertuples()]
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

        # Cache to parquet
        gdf.to_parquet(cache_path)
        print(f"  DINS: {len(gdf):,} primary structures — cached → {cache_path.name}")
        return gdf

    except Exception as e:
        print(f"  DINS API failed: {e}")
        # ── Local fallback ─────────────────────────────────────────────
        if cache_path.exists():
            print(f"  Falling back to cached file: {cache_path}")
            gdf = gpd.read_parquet(cache_path)
            print(f"  DINS: {len(gdf):,} structures (local fallback)")
            return gdf

        alt_paths = list(cache_dir.parent.glob(f'**/*DINS*{dins_name}*.parquet'))
        if alt_paths:
            print(f"  Found local DINS file: {alt_paths[0]}")
            gdf = gpd.read_parquet(alt_paths[0])
            if 'STRUCTUREC' in gdf.columns:
                gdf = gdf[~gdf['STRUCTUREC'].isin(DINS_EXCLUDE)].copy()
            if 'geometry' not in gdf.columns or gdf.crs is None:
                gdf['geometry'] = [Point(r.LONGITUDE, r.LATITUDE)
                                   for r in gdf.itertuples()]
                gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
            return gdf

        print(f"  No DINS data available for '{dins_name}'")
        return None

---
## 3. Fire Perimeter — CAL FIRE FRAP

Fetches the final burn perimeter polygon from the CAL FIRE Fire Perimeters  
all-years layer.  The returned geometry is the largest polygon for the  
matching fire (handles multi-polygon responses by taking the max area).

The perimeter bbox — expanded by ~500 m — drives both the ohsome and  
Overture spatial queries in the next two sections.


In [ ]:
def _esri_rings_to_shapely(esri_geom):
    """
    Convert an ArcGIS JSON polygon geometry (dict with 'rings') to Shapely.
    Handles multi-ring responses via unary_union.
    """
    from shapely.geometry import Polygon
    from shapely.ops import unary_union
    rings = (esri_geom or {}).get('rings', [])
    if not rings:
        return None
    polys = []
    for ring in rings:
        try:
            p = Polygon(ring)
            if p.is_valid and not p.is_empty:
                polys.append(p)
        except Exception:
            pass
    if not polys:
        return None
    return unary_union(polys)


def fetch_fire_perimeter(where_clause: str):
    """
    Fetch the final CAL FIRE burn perimeter polygon.

    Uses f=json (ArcGIS native JSON) with outSR=4326 so coordinates come
    back in WGS84.  The Esri ring geometry is converted to Shapely via
    _esri_rings_to_shapely.

    Parameters
    ----------
    where_clause : SQL filter, e.g. "FIRE_NAME='DIXIE' AND YEAR_=2021"

    Returns
    -------
    (perimeter_shapely_geom, bbox_tuple)
    bbox = (xmin, ymin, xmax, ymax) in WGS84, with 0.005 deg (~500 m) buffer
    """
    params = {
        'where':          where_clause,
        'outFields':      'FIRE_NAME,YEAR_,GIS_ACRES,ALARM_DATE',
        'returnGeometry': 'true',
        'outSR':          '4326',
        'f':              'json',
    }
    # Try primary URL, fall back to mirror if the service returns an error
    for url_attempt in [CALFIRE_URL, CALFIRE_URL_FALLBACK]:
        resp = requests.get(url_attempt, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if 'error' not in data:
            print(f'  Using perimeter service: {url_attempt.split("/arcgis")[0]}')
            break
        print(f'  {url_attempt.split("/arcgis")[0]} returned error: {data["error"].get("message","")} — trying fallback...')
    else:
        raise RuntimeError(f"ArcGIS error: {data['error']}")

    feats = data.get('features', [])
    if not feats:
        raise RuntimeError(f'No CAL FIRE perimeter features for: {where_clause}')

    geoms = [_esri_rings_to_shapely(f.get('geometry'))
             for f in feats if f.get('geometry')]
    geoms = [g for g in geoms if g is not None]
    if not geoms:
        raise RuntimeError(f'Could not parse geometry for: {where_clause}')

    perimeter = max(geoms, key=lambda g: g.area)

    pb   = perimeter.bounds
    bbox = (pb[0]-0.005, pb[1]-0.005, pb[2]+0.005, pb[3]+0.005)

    print(f'  Perimeter area: {perimeter.area * 111**2:.0f} km2')
    print(f'  Query bbox: {bbox}')
    return perimeter, bbox

---
## 4. Pre-Fire OSM Buildings — ohsome API

The [ohsome API](https://api.ohsome.org) (HeiGIT, Heidelberg University)  
reconstructs any past OSM snapshot on demand from the full edit history.

- Endpoint: `POST /elements/geometry`
- `time` parameter: fire date − 1 day → returns OSM state as of that date
- Filter: `building=* and geometry:polygon`
- No API key required; rate-limited but generous for research

The timestamp field (`@snapshotTimestamp`) is auto-detected from the first  
feature's property keys for robustness across API versions.


In [ ]:
def fetch_osm_buildings(bbox: tuple, pre_fire_date: str) -> gpd.GeoDataFrame:
    """
    Query ohsome API for all OSM buildings as of pre_fire_date.

    Parameters
    ----------
    bbox          : (xmin, ymin, xmax, ymax) WGS84
    pre_fire_date : 'YYYY-MM-DD' — date to snapshot OSM state

    Returns
    -------
    GeoDataFrame (EPSG:4326) with source='openstreetmap' and source_date column.
    """
    xmin, ymin, xmax, ymax = bbox
    params = {
        'bboxes':     f'{xmin},{ymin},{xmax},{ymax}',
        'filter':     'building=* and geometry:polygon',
        'time':       pre_fire_date,
        'properties': 'tags,metadata',
    }
    print(f"  ohsome query date: {pre_fire_date}")
    resp = requests.post(OHSOME_URL, data=params, timeout=300)

    if resp.status_code != 200:
        print(f"  ohsome returned {resp.status_code} — returning empty GDF")
        return gpd.GeoDataFrame(columns=['geometry','source','source_date'],
                                crs='EPSG:4326')

    fc     = resp.json()
    n_feat = len(fc.get('features', []))
    print(f"  ohsome: {n_feat:,} features returned")
    if n_feat == 0:
        return gpd.GeoDataFrame(columns=['geometry','source','source_date'],
                                crs='EPSG:4326')

    # Auto-detect the timestamp field (varies by API version)
    sample   = fc['features'][0].get('properties', {})
    ts_key   = next((k for k in ['@snapshotTimestamp','@timestamp',
                                  '@validFrom','timestamp']
                     if k in sample), None)
    print(f"  Timestamp field: {ts_key!r}")

    records = []
    for feat in fc.get('features', []):
        props = feat.get('properties') or {}
        try:
            geom = shape(feat['geometry'])
        except Exception:
            continue
        if not geom.is_valid:
            geom = geom.buffer(0)
        if geom.is_empty or not geom.is_valid:
            continue

        osm_id = props.get('@osmId', '')
        records.append({
            'geometry':    geom,
            'source':      'openstreetmap',
            'source_date': props.get(ts_key) if ts_key else None,
            'osm_id':      osm_id.split('/')[-1] if '/' in osm_id else osm_id,
            'building':    props.get('building', 'yes'),
            'name':        props.get('name'),
            'height':      props.get('height'),
            'levels':      props.get('building:levels'),
        })

    gdf = gpd.GeoDataFrame(records, crs='EPSG:4326')
    gdf['source_date'] = pd.to_datetime(gdf['source_date'], utc=True, errors='coerce')
    n_dated = gdf['source_date'].notna().sum()
    print(f"  OSM: {len(gdf):,} buildings  ({n_dated:,} with timestamps)")
    return gdf


---
## 5. ML Building Footprints — Overture Maps (S3)

Overture Maps ingests and conflates multiple ML building datasets  
(Microsoft, Google Open Buildings, Esri, etc.) and publishes monthly  
releases on a public S3 bucket in Apache Parquet format.

We keep **all non-OSM primary sources** — the OSM buildings are already  
captured with precise dating via ohsome, so including them again from  
Overture would cause duplicates.

**Important caveat:** Overture uses *current* imagery, so for areas with  
heavy fire destruction and limited rebuilding, ML coverage will undercount  
pre-fire structures.  This is corrected in Section 8 using DINS.


In [ ]:
def fetch_overture_buildings(bbox: tuple) -> gpd.GeoDataFrame:
    """
    Download non-OSM ML buildings from the latest Overture Maps release.

    Uses anonymous S3 access — no credentials required.
    pyarrow bbox filter prunes row groups efficiently before download.

    Parameters
    ----------
    bbox : (xmin, ymin, xmax, ymax) WGS84

    Returns
    -------
    GeoDataFrame (EPSG:4326) with source = Overture dataset name.
    """
    import pyarrow.compute as pc
    import pyarrow.dataset as ds
    import pyarrow.fs as pafs
    from urllib.request import urlopen

    xmin, ymin, xmax, ymax = bbox

    # Discover latest Overture release from the STAC catalog
    with urlopen('https://stac.overturemaps.org/catalog.json') as r:
        latest = json.load(r)['latest']
    print(f"  Overture release: {latest}")

    s3_path = (f"overturemaps-us-west-2/release/{latest}/"
               "theme=buildings/type=building/")
    s3      = pafs.S3FileSystem(anonymous=True, region='us-west-2')
    dataset = ds.dataset(s3_path, filesystem=s3)

    # Spatial pre-filter using the bbox columns in the Parquet schema
    bbox_filter = (
        (pc.field('bbox','xmin') < xmax) & (pc.field('bbox','xmax') > xmin) &
        (pc.field('bbox','ymin') < ymax) & (pc.field('bbox','ymax') > ymin)
    )
    table = dataset.to_table(filter=bbox_filter)
    print(f"  Overture: {table.num_rows:,} rows in bbox")

    records = []
    tally   = {}
    for row in table.to_pylist():
        sources = row.get('sources') or []
        if not sources:
            continue
        primary = sources[0].get('dataset', '').lower()
        if 'openstreetmap' in primary:
            continue          # skip OSM — already captured via ohsome

        try:
            geom = wkb.loads(bytes(row['geometry']))
        except Exception:
            continue
        if geom is None or geom.is_empty:
            continue
        if not geom.is_valid:
            geom = geom.buffer(0)

        records.append({
            'geometry':    geom,
            'source':      primary,
            'source_date': pd.NaT,
            'osm_id':      None,
            'building':    'yes',
            'name':        None,
            'height':      None,
            'levels':      None,
        })
        tally[primary] = tally.get(primary, 0) + 1

    print(f"  Non-OSM ML buildings: {len(records):,}")
    for src, cnt in sorted(tally.items(), key=lambda x: -x[1]):
        print(f"    {src:<45} {cnt:>6,}")

    if not records:
        return gpd.GeoDataFrame(columns=['geometry','source','source_date'],
                                crs='EPSG:4326')
    return gpd.GeoDataFrame(records, crs='EPSG:4326')


---
## 6. Merge, Deduplicate & Clip

**IoU deduplication:** Any ML building whose footprint overlaps an OSM  
footprint by more than `IOU_THRESHOLD` (default 30%) is removed — it  
is already represented by the more accurately dated OSM polygon.

**Clip to perimeter:** Only buildings inside the burn perimeter are kept.  
Buildings just outside (in the 500 m query buffer) are discarded.


In [ ]:
def remove_ml_duplicates(
    gdf_osm: gpd.GeoDataFrame,
    gdf_ml:  gpd.GeoDataFrame,
    iou_threshold: float = IOU_THRESHOLD,
) -> gpd.GeoDataFrame:
    """
    Drop ML buildings that substantially overlap an OSM footprint.

    Uses a spatial index (sindex) so only nearby OSM buildings are tested.
    IoU = intersection_area / union_area; drop ML building if IoU >= threshold.
    """
    if len(gdf_ml) == 0 or len(gdf_osm) == 0:
        return gdf_ml

    sindex = gdf_osm.sindex
    keep   = np.ones(len(gdf_ml), dtype=bool)

    for i, row in enumerate(gdf_ml.itertuples()):
        geom = row.geometry
        if geom is None or geom.is_empty:
            keep[i] = False
            continue
        for j in sindex.intersection(geom.bounds):
            og = gdf_osm.iloc[j].geometry
            if not geom.intersects(og):
                continue
            inter = geom.intersection(og).area
            union = geom.union(og).area
            if union > 0 and (inter / union) >= iou_threshold:
                keep[i] = False
                break

    kept = gdf_ml[keep].reset_index(drop=True)
    print(f"  IoU dedup: dropped {(~keep).sum():,}, kept {keep.sum():,} ML buildings")
    return kept


def merge_and_clip(
    gdf_osm:   gpd.GeoDataFrame,
    gdf_ml:    gpd.GeoDataFrame,
    perimeter,
) -> gpd.GeoDataFrame:
    """
    Deduplicate, concatenate, clip to perimeter, remove tiny polygons.
    Returns GeoDataFrame (EPSG:4326).
    """
    gdf_ml_dd = remove_ml_duplicates(gdf_osm, gdf_ml)

    gdf_all = pd.concat([gdf_osm, gdf_ml_dd], ignore_index=True)
    gdf_all = gpd.GeoDataFrame(gdf_all, crs='EPSG:4326')
    gdf_all = gdf_all[
        gdf_all.geometry.notna() &
        ~gdf_all.geometry.is_empty &
        gdf_all.geometry.is_valid
    ].copy()

    fire_gdf = gpd.GeoDataFrame(geometry=[perimeter], crs='EPSG:4326')
    gdf_all  = gpd.clip(gdf_all, fire_gdf).reset_index(drop=True)

    print(f"  After clip: {len(gdf_all):,} buildings inside perimeter")
    print(gdf_all['source'].value_counts().to_string())
    return gdf_all


---
## 7. Spatial Metrics

Two building separation metrics are computed for every footprint:

| Metric | Definition | Fire science relevance |
|--------|-----------|----------------------|
| `min_wall_wall_m` | Minimum exterior gap between polygon walls (Shapely `.distance()`) | Directly governs radiant heat flux; used as primary input to SSDD |
| `nn_dist_m` | Centroid-to-centroid nearest-neighbour distance | Comparable to legacy literature metrics |

Reference thresholds from WUI fire research:
- **< 3 m** — structures nearly touching; near-certain fire jump
- **< 7.6 m** — high radiant ignition probability (1.5× typical building width)
- **< 15 m** — elevated risk zone for direct flame impingement

`orientation_deg` is the long-axis angle of the minimum rotated rectangle  
(0–180°), used by SSDD's orientation weighting term.


In [ ]:
def _mbr_orientation(geom) -> float:
    """Long-axis orientation of the minimum rotated rectangle, 0–180°."""
    try:
        coords = list(geom.minimum_rotated_rectangle.exterior.coords)
        edges  = [(math.hypot(coords[i+1][0]-coords[i][0],
                              coords[i+1][1]-coords[i][1]),
                   coords[i+1][0]-coords[i][0],
                   coords[i+1][1]-coords[i][1])
                  for i in range(len(coords)-1)]
        _, dx, dy = max(edges, key=lambda e: e[0])
        return math.degrees(math.atan2(dy, dx)) % 180
    except Exception:
        return np.nan


def compute_spatial_metrics(
    gdf_utm: gpd.GeoDataFrame,
    search_r: float = WALL_SEARCH_R,
) -> gpd.GeoDataFrame:
    """
    Add spatial metric columns to a UTM-projected GeoDataFrame:
      area_m2         — footprint area in m²
      orientation_deg — MRR long-axis angle (0–180°)
      min_wall_wall_m — minimum wall-to-wall exterior gap to any neighbour
                        within search_r metres (NaN if no neighbour found)
      nn_dist_m       — centroid-to-centroid nearest-neighbour distance

    Parameters
    ----------
    gdf_utm  : GeoDataFrame projected to a metric CRS (e.g. EPSG:32610)
    search_r : wall-to-wall neighbour search radius in metres
    """
    gdf = gdf_utm.copy()
    gdf['area_m2'] = gdf.geometry.area

    # Orientation — MRR long-axis angle
    print("  Computing orientations…")
    gdf['orientation_deg'] = [_mbr_orientation(g) for g in gdf.geometry]

    polys = gdf.geometry.values
    pts   = np.array([[g.centroid.x, g.centroid.y] for g in polys])

    # Guard: need at least 2 buildings for meaningful distance metrics
    if len(gdf) < 2:
        print(f'  Only {len(gdf)} building(s) in perimeter — skipping distance metrics')
        gdf['nn_dist_m'] = np.nan
        gdf['min_wall_wall_m'] = np.nan
        return gdf

    # Centroid-to-centroid: fast via KDTree (k=2 to exclude self)
    kd_dists, _ = cKDTree(pts).query(pts, k=2)
    gdf['nn_dist_m'] = kd_dists[:, 1]
    gdf.loc[np.isinf(gdf['nn_dist_m']), 'nn_dist_m'] = np.nan

    # Wall-to-wall: STRtree spatial prefilter, then exact polygon distance
    # Shapely polygon.distance(other) = minimum gap between exterior rings
    print(f"  Computing wall-to-wall gaps (search_r={search_r:.0f} m)…")
    tree    = STRtree(polys)
    min_ww  = np.full(len(gdf), np.inf)

    for i in tqdm(range(len(gdf)), desc='wall-to-wall', leave=False):
        Pi    = polys[i]
        cands = tree.query(Pi.buffer(search_r))
        for j in cands:
            if j == i:
                continue
            d = Pi.distance(polys[j])
            if d < min_ww[i]:
                min_ww[i] = d

    min_ww[np.isinf(min_ww)] = np.nan
    gdf['min_wall_wall_m'] = min_ww

    # Print summary stats
    ww = gdf['min_wall_wall_m'].dropna()
    print(f"  Wall-to-wall  median={np.median(ww):.1f} m  "
          f"p10={np.percentile(ww,10):.1f} m  "
          f"<3m: {(ww<3).mean()*100:.1f}%  "
          f"<15m: {(ww<15).mean()*100:.1f}%")
    return gdf

---
## 8. DINS-Centric Unified Dataset

**Why this matters:** Overture ML buildings are derived from *current* satellite  
imagery.  For communities destroyed by fire and never rebuilt, the ML model  
simply cannot detect them — producing a systematic post-fire survivorship bias  
(the Camp Fire showed only 39% DINS match rate with Overture alone).

**Solution:** Treat DINS as the authoritative pre-fire structure inventory.  
For each DINS structure:

- If a footprint polygon centroid is within `MATCH_M` metres → use the real polygon  
- Otherwise → create a circular footprint sized from the median area of matched  
  buildings in the same `STRUCTUREC` class (Single Residence, Commercial, etc.)

Every record gets a `geometry_source` flag:
- `matched_footprint` — real ML/OSM polygon geometry
- `estimated_centroid` — circular proxy from DINS location + class median area

The unified dataset is what you should feed into the SSDD notebook.


In [ ]:
def build_dins_hybrid(
    gdf_all:  gpd.GeoDataFrame,   # merged ML+OSM in WGS84
    gdf_dins: gpd.GeoDataFrame,   # DINS primary structures in WGS84
    match_m:  float = MATCH_M,
) -> tuple:
    """
    Build a DINS-centric unified structure dataset.

    Priority order per DINS point:
      1. Nearest footprint centroid within match_m → use real polygon
      2. No nearby footprint → circular estimate from class-median area

    Parameters
    ----------
    gdf_all  : merged OSM + ML footprints (EPSG:4326)
    gdf_dins : filtered DINS primary structures (EPSG:4326)
    match_m  : centroid match threshold in metres

    Returns
    -------
    (gdf_unified, match_pct)
      gdf_unified : GeoDataFrame (EPSG:32610) with geometry_source flag
      match_pct   : float, percentage of DINS points matched to real footprint
    """
    # Project both to UTM Zone 10N (metres)
    gdf_fp = gdf_all.to_crs('EPSG:32610').copy()
    gdf_fp['area_m2'] = gdf_fp.geometry.area
    gdf_d  = gdf_dins.to_crs('EPSG:32610').copy()

    # KDTree on footprint centroids
    fp_cents = np.array([[g.centroid.x, g.centroid.y] for g in gdf_fp.geometry])
    d_pts    = np.array([[g.x, g.y] for g in gdf_d.geometry])
    nn_dists, nn_idx = cKDTree(fp_cents).query(d_pts, k=1)

    # Compute per-class median footprint area from matched buildings only
    # (avoids using post-fire-biased ML sizes for estimated footprints)
    struct_areas = {}
    for i, row in enumerate(gdf_d.itertuples()):
        if nn_dists[i] <= match_m:
            struct_areas.setdefault(row.STRUCTUREC, []).append(
                float(gdf_fp.iloc[nn_idx[i]]['area_m2'])
            )
    struct_medians = {cls: float(np.median(v)) for cls, v in struct_areas.items()}
    overall_median = (float(np.median([a for v in struct_areas.values() for a in v]))
                      if struct_areas else 100.0)

    print(f"  Median footprint area by structure class (from matched buildings):")
    for cls, med in sorted(struct_medians.items(), key=lambda x: -x[1]):
        n = len(struct_areas[cls])
        print(f"    {cls:<35} {med:>7.0f} m²  (n={n:,})")
    print(f"    {'Overall fallback':<35} {overall_median:>7.0f} m²")

    # Assemble unified dataset
    records   = []
    n_matched = n_est = 0

    for i, row in enumerate(gdf_d.itertuples()):
        dist, fp_i = float(nn_dists[i]), int(nn_idx[i])
        if dist <= match_m:
            fp      = gdf_fp.iloc[fp_i]
            geom    = fp.geometry
            area    = float(fp['area_m2'])
            gsrc    = 'matched_footprint'
            fpsrc   = fp['source']
            n_matched += 1
        else:
            med    = struct_medians.get(row.STRUCTUREC, overall_median)
            geom   = row.geometry.buffer(math.sqrt(med / math.pi))
            area   = med
            gsrc   = 'estimated_centroid'
            fpsrc  = 'estimated'
            n_est += 1

        records.append({
            'geometry':        geom,
            'STRUCTUREC':      row.STRUCTUREC,
            'DAMAGE':          getattr(row, 'DAMAGE', None),
            'LATITUDE':        row.LATITUDE,
            'LONGITUDE':       row.LONGITUDE,
            'area_m2':         area,
            'fp_source':       fpsrc,
            'geometry_source': gsrc,
            'nn_dist_m':       dist,
        })

    gdf_u     = gpd.GeoDataFrame(records, crs='EPSG:32610')
    match_pct = n_matched / max(len(gdf_u), 1) * 100
    print(f"  Unified: {len(gdf_u):,} structures — "
          f"matched {n_matched:,} ({match_pct:.1f}%), "
          f"estimated {n_est:,} ({100-match_pct:.1f}%)")
    return gdf_u, match_pct


---
## 9. Visualisation Helpers

In [ ]:
SOURCE_COLORS = {
    'openstreetmap':         '#2196F3',
    'microsoft-buildings':   '#FF6B35',
    'google-open-buildings': '#4CAF50',
    'esri-buildings':        '#9C27B0',
}
def _get_color(src):
    src = (src or '').lower()
    for k, c in SOURCE_COLORS.items():
        if k in src: return c
    return '#9E9E9E'


def save_source_map(gdf_all, perimeter, path):
    """Static map coloured by data source with fire perimeter overlay."""
    fig, ax = plt.subplots(figsize=(12, 8))
    gdf_all.plot(ax=ax, color=gdf_all['source'].apply(_get_color),
                 edgecolor='none', alpha=0.7)
    gpd.GeoSeries([perimeter]).plot(ax=ax, facecolor='none',
                                    edgecolor='red', linewidth=1, linestyle='--')
    patches = [mpatches.Patch(color=_get_color(s), label=f'{s} ({n:,})')
               for s, n in gdf_all['source'].value_counts().items()]
    patches.append(mpatches.Patch(facecolor='none', edgecolor='red',
                                  linestyle='--', label='Burn perimeter'))
    ax.legend(handles=patches, loc='upper right', fontsize=8)
    ax.set_title('Pre-Fire Buildings — Data Source', fontsize=11)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {path.name}")


def save_spacing_plot(gdf_utm, path, fire_name):
    """Side-by-side wall-to-wall and centroid spacing histograms."""
    ww = gdf_utm['min_wall_wall_m'].dropna()
    cc = gdf_utm['nn_dist_m'].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(ww[ww < 60], bins=60, color='#E53935',
                 edgecolor='white', alpha=0.85)
    axes[0].axvline(np.median(ww), color='darkred', linestyle='--',
                    linewidth=1.5, label=f'Median: {np.median(ww):.1f} m')
    for x, lbl, clr in [(3, '3 m', 'orange'), (7.6, '7.6 m', 'gold'),
                         (15, '15 m', 'yellow')]:
        axes[0].axvline(x, color=clr, linestyle=':', linewidth=1.2, label=lbl)
    axes[0].set_xlabel('Min wall-to-wall gap (m)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Wall-to-Wall Separation (primary metric)')
    axes[0].legend(fontsize=7)

    axes[1].hist(cc[cc < 120], bins=60, color='coral',
                 edgecolor='white', alpha=0.85)
    axes[1].axvline(np.median(cc), color='darkred', linestyle='--',
                    linewidth=1.5, label=f'Median: {np.median(cc):.1f} m')
    axes[1].set_xlabel('Centroid-to-centroid distance (m)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Centroid Spacing (reference)')
    axes[1].legend(fontsize=8)

    fig.suptitle(f"Building Separation — {fire_name.replace('_',' ')} (pre-fire)",
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {path.name}")


---
## 10. Per-Fire Pipeline Orchestrator

`process_fire()` runs the full 7-step pipeline for one fire entry  
from `FIRE_INVENTORY` and writes all outputs to `OUTPUT_ROOT/<slug>/`.

Steps:
1. Fetch CAL FIRE burn perimeter → derive query bbox  
2. Fetch OSM buildings at pre-fire date (ohsome)  
3. Fetch ML buildings (Overture S3)  
4. Merge + IoU dedup + clip to perimeter  
5. Project to UTM, compute wall-to-wall + orientation metrics  
6. Fetch DINS from live API (or cache); build DINS-centric unified dataset  
7. Save GeoPackages, QA plots, summary JSON

Set `resume=True` to skip fires whose `_summary.json` already shows `status: success`.


In [ ]:
def process_fire(fire: dict, resume: bool = False) -> dict:
    """
    Full pipeline for one fire.  Returns a summary dict.

    Parameters
    ----------
    fire   : one entry from FIRE_INVENTORY
    resume : if True, skip fires with an existing success summary

    Returns
    -------
    dict with status, counts, metrics, and output file paths
    """
    slug    = fire['name'].lower()
    out_dir = OUTPUT_ROOT / slug
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_path = out_dir / f"{slug}_summary.json"

    # Resume check
    if resume and summary_path.exists():
        saved = json.loads(summary_path.read_text())
        if saved.get('status') == 'success':
            print(f"[{fire['name']}] already completed — skipping (resume=True)")
            return saved

    t0      = time.time()
    summary = {
        'fire':          fire['name'],
        'ignition_date': fire['date'],
        'processed_at':  datetime.utcnow().isoformat(),
        'status':        'failed',
    }

    try:
        print(f"\n{'='*60}")
        print(f"  FIRE: {fire['name']}  ({fire['date']})")
        print(f"{'='*60}")

        # ── 1. Perimeter ──────────────────────────────────────────────
        print("\n[1/7] Fetching CAL FIRE perimeter…")
        perimeter, bbox = fetch_fire_perimeter(fire['where'])

        # ── 2. OSM ────────────────────────────────────────────────────
        print("\n[2/7] Fetching pre-fire OSM buildings (ohsome)…")
        pre_date = (datetime.strptime(fire['date'], '%Y-%m-%d')
                    - timedelta(days=1)).strftime('%Y-%m-%d')
        gdf_osm  = fetch_osm_buildings(bbox, pre_date)

        # ── 3. Overture ML ────────────────────────────────────────────
        print("\n[3/7] Fetching ML buildings (Overture S3)…")
        gdf_ml = fetch_overture_buildings(bbox)

        # ── 4. Merge + clip ───────────────────────────────────────────
        print("\n[4/7] Merging and clipping to perimeter…")
        gdf_all = merge_and_clip(gdf_osm, gdf_ml, perimeter)
        gdf_all['source_date'] = pd.to_datetime(
            gdf_all.get('source_date'), utc=True, errors='coerce')

        # ── 5. Spatial metrics ────────────────────────────────────────
        print("\n[5/7] Computing spatial metrics…")
        gdf_utm = gdf_all.to_crs('EPSG:32610').copy()
        gdf_utm = gdf_utm[gdf_utm.geometry.area >= MIN_AREA_M2].copy()
        gdf_utm = compute_spatial_metrics(gdf_utm)

        # ── 6. DINS hybrid ────────────────────────────────────────────
        print("\n[6/7] Fetching DINS and building unified dataset…")
        gdf_dins  = fetch_dins_for_fire(fire.get('dins_name'), DINS_CACHE_DIR,
                                        where_override=fire.get('dins_where'))
        match_pct = None

        if gdf_dins is not None and len(gdf_dins) > 0:
            print(f"  DINS: {len(gdf_dins):,} primary structures")
            gdf_unified, match_pct = build_dins_hybrid(gdf_all, gdf_dins)
            # Add wall-to-wall metrics to unified dataset too
            gdf_unified = compute_spatial_metrics(gdf_unified)
            unified_path = out_dir / f"{slug}_unified_structures.gpkg"
            gdf_unified.to_crs('EPSG:4326').to_file(unified_path, driver='GPKG')
            print(f"  Unified dataset saved → {unified_path.name}")
        else:
            unified_path = None
            print("  No DINS data — unified dataset not created")

        # ── 7. Save ───────────────────────────────────────────────────
        print("\n[7/7] Saving outputs…")

        # Transfer metrics back into WGS84 GDF for the main save
        metric_cols = ['area_m2', 'orientation_deg', 'min_wall_wall_m', 'nn_dist_m']
        gdf_save = gdf_all.copy()
        for col in metric_cols:
            if col in gdf_utm.columns:
                gdf_save = gdf_save.merge(
                    gdf_utm[[col]].reset_index(),
                    left_index=True, right_on='index', how='left'
                ).drop(columns='index', errors='ignore')

        gpkg_path = out_dir / f"{slug}_prefire_buildings.gpkg"
        gdf_save.to_file(gpkg_path, driver='GPKG')
        print(f"  Footprints saved → {gpkg_path.name}")

        save_source_map(gdf_all, perimeter, out_dir / f"{slug}_source_map.png")
        save_spacing_plot(gdf_utm, out_dir / f"{slug}_spacing.png", fire['name'])

        # Build summary
        ww = gdf_utm['min_wall_wall_m'].dropna() if 'min_wall_wall_m' in gdf_utm.columns else pd.Series([], dtype=float)
        elapsed = time.time() - t0
        summary.update({
            'status':             'success',
            'elapsed_sec':        round(elapsed, 1),
            'perimeter_km2':      round(perimeter.area * 111**2, 1),
            'pre_fire_date':      pre_date,
            'n_buildings_merged': len(gdf_all),
            'n_buildings_utm':    len(gdf_utm),
            'source_counts':      gdf_all['source'].value_counts().to_dict(),
            'ww_median_m':        round(float(np.median(ww)), 1) if len(ww) else None,
            'ww_p10_m':           round(float(np.percentile(ww, 10)), 1) if len(ww) else None,
            'ww_lt3m_pct':        round(float((ww < 3).mean() * 100), 1) if len(ww) else None,
            'ww_lt15m_pct':       round(float((ww < 15).mean() * 100), 1) if len(ww) else None,
            'dins_n_structures':  len(gdf_dins) if gdf_dins is not None else None,
            'dins_match_pct':     round(match_pct, 1) if match_pct else None,
            'output_gpkg':        str(gpkg_path),
            'output_unified':     str(unified_path) if unified_path else None,
        })
        print(f"\n  ✓  {fire['name']} complete in {elapsed:.0f} s")

    except Exception as exc:
        import traceback
        print(f"\n  ✗  {fire['name']} FAILED: {exc}")
        traceback.print_exc()
        summary['error'] = str(exc)

    summary_path.write_text(json.dumps(summary, indent=2))
    return summary

---
## 11. Run — Single Fire (Interactive)

Set `CURRENT_FIRE` to any name from `FIRE_INVENTORY`, then run this cell.  
Use this to test the pipeline on one fire before running the full batch.


In [ ]:
# ── Select the fire to process ────────────────────────────────────────
# Change this to any name from FIRE_INVENTORY above, e.g.:
#   'Woolsey', 'Kincade', 'Dixie', 'Caldor', 'Park', 'Palisades', 'Eaton' …
CURRENT_FIRE = 'Dixie'

# ── Run ───────────────────────────────────────────────────────────────
fire_cfg = next((f for f in FIRE_INVENTORY if f['name'] == CURRENT_FIRE), None)
if fire_cfg is None:
    print(f"Fire '{CURRENT_FIRE}' not found.  Available names:")
    for f in FIRE_INVENTORY:
        print(f"  {f['name']}")
else:
    result = process_fire(fire_cfg, resume=False)
    print()
    print("── Summary ──────────────────────────────────────────────")
    for k, v in result.items():
        if k not in ('source_counts',):
            print(f"  {k:<25} {v}")


---
## 12. Run — All Fires (Batch)

Processes every fire in `FIRE_INVENTORY` sequentially.  
Set `RESUME = True` to skip fires that already have a `success` summary JSON  
(useful if a run is interrupted partway through).

**Expected runtime:** 5–20 minutes per fire depending on perimeter size  
and the wall-to-wall computation (the bottleneck for large fires like Dixie).


In [ ]:
# ── Batch configuration ───────────────────────────────────────────────
RESUME      = True   # skip already-completed fires
FIRE_SUBSET = None   # set to a list of names to run only those, e.g.:
                     # ['Woolsey', 'Glass', 'Caldor']
                     # Leave as None to run all fires

# ── Run ───────────────────────────────────────────────────────────────
fires_to_run = FIRE_INVENTORY
if FIRE_SUBSET:
    fires_to_run = [f for f in FIRE_INVENTORY if f['name'] in FIRE_SUBSET]

print(f"Processing {len(fires_to_run)} fire(s)…  resume={RESUME}\n")

all_summaries = []
for fire in fires_to_run:
    s = process_fire(fire, resume=RESUME)
    all_summaries.append(s)

# Save batch summary CSV
df_sum = pd.DataFrame(all_summaries)
csv_path = OUTPUT_ROOT / 'batch_summary.csv'
df_sum.to_csv(csv_path, index=False)
print(f"\nBatch summary → {csv_path}")


---
## 13. DINS Patch — Re-run DINS Only

Use this cell to backfill DINS data for fires that already have a successful
`_prefire_buildings.gpkg` without repeating the expensive OSM / Overture /
wall-to-wall steps.

For each fire it:
1. Loads the existing `_prefire_buildings.gpkg` (WGS84)
2. Fetches DINS fresh from the CAL FIRE API (respects `DINS_FORCE_REFRESH`)
3. Runs `build_dins_hybrid` + `compute_spatial_metrics` on the unified dataset
4. Saves `_unified_structures.gpkg`
5. Patches only the DINS-related fields in `_summary.json`

Fires that have no existing GPKG are skipped with a warning.

In [ ]:
# ── DINS Patch configuration ───────────────────────────────────────────
# Fires to patch — leave as None to patch all fires in FIRE_INVENTORY
# (fires without an existing _prefire_buildings.gpkg are automatically skipped)
DINS_PATCH_SUBSET = None   # e.g. ['Woolsey', 'Glass'] to patch only those

# ── Run ───────────────────────────────────────────────────────────────
patch_fires = FIRE_INVENTORY
if DINS_PATCH_SUBSET:
    patch_fires = [f for f in FIRE_INVENTORY if f['name'] in DINS_PATCH_SUBSET]

patch_results = []

for fire in patch_fires:
    slug      = fire['name'].lower()
    out_dir   = OUTPUT_ROOT / slug
    gpkg_path = out_dir / f"{slug}_prefire_buildings.gpkg"
    summary_path = out_dir / f"{slug}_summary.json"

    print(f"\n{'='*55}")
    print(f"  DINS PATCH: {fire['name']}")
    print(f"{'='*55}")

    if not gpkg_path.exists():
        print(f"  SKIP — no prefire buildings GPKG found: {gpkg_path}")
        patch_results.append({'fire': fire['name'], 'patch_status': 'skipped_no_gpkg'})
        continue

    dins_name = fire.get('dins_name')
    if dins_name is None:
        print(f"  SKIP — no dins_name configured for this fire")
        patch_results.append({'fire': fire['name'], 'patch_status': 'skipped_no_dins_name'})
        continue

    try:
        # Load existing footprints
        gdf_all = gpd.read_file(gpkg_path)
        if gdf_all.crs is None or gdf_all.crs.to_epsg() != 4326:
            gdf_all = gdf_all.to_crs('EPSG:4326')
        print(f"  Loaded {len(gdf_all):,} footprints from {gpkg_path.name}")

        # Fetch DINS fresh (honours DINS_FORCE_REFRESH; pass exact WHERE if configured)
        gdf_dins = fetch_dins_for_fire(dins_name, DINS_CACHE_DIR,
                                       where_override=fire.get('dins_where'))

        if gdf_dins is None or len(gdf_dins) == 0:
            print(f"  No DINS data returned — skipping unified build")
            patch_results.append({'fire': fire['name'], 'patch_status': 'no_dins_data',
                                  'dins_n': 0})
            continue

        if len(gdf_all) < 2:
            print(f"  Only {len(gdf_all)} footprint(s) — cannot build meaningful hybrid")
            patch_results.append({'fire': fire['name'], 'patch_status': 'too_few_footprints',
                                  'dins_n': len(gdf_dins)})
            continue

        print(f"  DINS: {len(gdf_dins):,} primary structures")

        # Build DINS-centric unified dataset
        gdf_unified, match_pct = build_dins_hybrid(gdf_all, gdf_dins)

        # Spatial metrics on unified dataset
        gdf_unified = compute_spatial_metrics(gdf_unified)

        # Save unified GPKG
        unified_path = out_dir / f"{slug}_unified_structures.gpkg"
        gdf_unified.to_crs('EPSG:4326').to_file(unified_path, driver='GPKG')
        print(f"  Unified dataset saved → {unified_path.name}")

        # Patch summary JSON (update only DINS fields, preserve everything else)
        dins_patch = {
            'dins_n_structures': len(gdf_dins),
            'dins_match_pct':    round(match_pct, 1),
            'output_unified':    str(unified_path),
        }
        if summary_path.exists():
            saved = json.loads(summary_path.read_text())
            saved.update(dins_patch)
            summary_path.write_text(json.dumps(saved, indent=2))
            print(f"  Summary patched — dins_n={len(gdf_dins):,}  match={match_pct:.1f}%")
        else:
            print(f"  WARNING: no summary JSON found at {summary_path}")

        patch_results.append({'fire': fire['name'], 'patch_status': 'ok',
                              'dins_n': len(gdf_dins), 'dins_match_pct': round(match_pct, 1)})

    except Exception as exc:
        import traceback
        print(f"  PATCH FAILED: {exc}")
        traceback.print_exc()
        patch_results.append({'fire': fire['name'], 'patch_status': f'error: {exc}'})

# Rebuild batch_summary.csv to reflect patched values
csv_path = OUTPUT_ROOT / 'batch_summary.csv'
if csv_path.exists() and csv_path.stat().st_size > 0:
    try:
        df_sum = pd.read_csv(csv_path)
        for r in patch_results:
            if r.get('patch_status') == 'ok':
                mask = df_sum['fire'] == r['fire']
                df_sum.loc[mask, 'dins_n_structures'] = r['dins_n']
                df_sum.loc[mask, 'dins_match_pct']    = r['dins_match_pct']
        df_sum.to_csv(csv_path, index=False)
        print(f"\nBatch summary CSV updated → {csv_path}")
    except Exception as e:
        print(f"\nWARNING: could not update batch_summary.csv: {e}")
else:
    print(f"\nNo batch_summary.csv found or file is empty — skipping CSV update")

print("\n── Patch results ──────────────────────────────────────────────")
for r in patch_results:
    n   = r.get('dins_n', '-')
    pct = r.get('dins_match_pct', '-')
    print(f"  {r['fire']:<30} {r['patch_status']:<12}  "
          f"dins_n={n}  match={pct}%")

---
## 14. Geometric Confidence Tier

Assigns each fire a data quality tier based on how much of the DINS-centric
unified dataset relies on **real pre-fire polygon geometry** vs **estimated
circular footprints**.  This is the primary determinant of SSDD reliability.

### Why this matters for SSDD
Wall-to-wall separation distances computed on circular (estimated) footprints
are systematically different from polygon-to-polygon distances:
- Circular footprints use the per-class **median area** of matched buildings
- Small or irregular structures get standardised to that median → spacing bias
- High estimated-centroid fractions make SSDD values less comparable across fires

### Tier definitions

| Tier | Criterion | SSDD suitability |
|------|-----------|-----------------|
| **A — High confidence** | match ≥ 60 % | Real polygons dominate; use directly |
| **B — Moderate confidence** | match 30–59 % AND ≥ 500 matched | Substantial polygon coverage; use with uncertainty note |
| **C — Low confidence** | match < 30 % OR < 500 matched polygons | Primarily circular estimates; exploratory only |
| **D — Not usable** | < 2 footprints OR perimeter scope mismatch | Exclude from SSDD analysis |

### Open Topography LiDAR check
The OT catalog was queried for pre-fire point-cloud datasets overlapping each
Tier C / D fire.  No usable pre-fire LiDAR was found for any fire in the
inventory — the only overlapping OT surveys are 2012-2014 USFS vegetation
flights (too old, too low density for building extraction).  Tier C fires should
therefore be excluded from primary SSDD analysis unless an alternative pre-fire
footprint source is identified.

In [ ]:
def assign_confidence_tier(dins_match_pct, n_matched_abs, n_footprints,
                            perimeter_mismatch=False):
    """
    Assign a geometric confidence tier for SSDD analysis.

    Parameters
    ----------
    dins_match_pct    : float | None  — % of DINS structures matched to a real polygon
    n_matched_abs     : int   | None  — absolute count of matched footprints
    n_footprints      : int           — total footprints inside perimeter
    perimeter_mismatch: bool          — True for fires where DINS scope ≠ perimeter scope

    Returns
    -------
    (tier, reason) : str, str
    """
    if perimeter_mismatch or n_footprints < 2:
        return 'D', ('perimeter scope mismatch' if perimeter_mismatch
                     else f'only {n_footprints} footprint(s) inside perimeter')

    if dins_match_pct is None or n_matched_abs is None:
        return 'D', 'no DINS data'

    if dins_match_pct >= 60:
        return 'A', f'{dins_match_pct:.1f}% match — real polygons dominate'

    if dins_match_pct >= 30 and n_matched_abs >= 500:
        return 'B', (f'{dins_match_pct:.1f}% match, {n_matched_abs:,} matched polygons'
                     ' — substantial polygon coverage')

    if dins_match_pct >= 30:
        return 'C', (f'{dins_match_pct:.1f}% match but only {n_matched_abs:,} matched'
                     ' — limited absolute polygon coverage')

    return 'C', f'{dins_match_pct:.1f}% match — primarily estimated centroids'


# Fires where the DINS query scope is known to be broader than the perimeter used
PERIMETER_MISMATCH_FIRES = {'LNU_Lightning_Complex'}

# ── Classify all fires and patch summaries ─────────────────────────────
tier_records = []

for fire in FIRE_INVENTORY:
    slug         = fire['name'].lower()
    summary_path = OUTPUT_ROOT / slug / f"{slug}_summary.json"
    if not summary_path.exists():
        continue

    s             = json.loads(summary_path.read_text())
    match_pct     = s.get('dins_match_pct')
    dins_n        = s.get('dins_n_structures')
    n_footprints  = s.get('n_buildings_utm', s.get('n_buildings_merged', 0))
    n_matched     = (round(dins_n * match_pct / 100)
                     if (dins_n and match_pct is not None) else None)
    is_mismatch   = fire['name'] in PERIMETER_MISMATCH_FIRES

    tier, reason = assign_confidence_tier(
        match_pct, n_matched, n_footprints, is_mismatch)

    # Write tier back into the summary JSON
    s['confidence_tier']        = tier
    s['confidence_tier_reason'] = reason
    s['n_dins_matched_abs']     = n_matched
    summary_path.write_text(json.dumps(s, indent=2))

    tier_records.append({
        'fire':          fire['name'],
        'tier':          tier,
        'match_pct':     match_pct,
        'n_dins':        int(dins_n) if dins_n else None,
        'n_matched':     n_matched,
        'n_footprints':  n_footprints,
        'ww_med_m':      s.get('ww_median_m'),
        'reason':        reason,
    })

# ── Display ────────────────────────────────────────────────────────────
df_tier = pd.DataFrame(tier_records).set_index('fire')

TIER_LABELS = {
    'A': 'A — High confidence (SSDD reliable)',
    'B': 'B — Moderate confidence (use with uncertainty note)',
    'C': 'C — Low confidence (exploratory only)',
    'D': 'D — Not usable (exclude from SSDD)',
}

print("═" * 80)
print("  GEOMETRIC CONFIDENCE TIERS — SSDD Analysis Suitability")
print("═" * 80)

for tier in ['A', 'B', 'C', 'D']:
    subset = df_tier[df_tier['tier'] == tier]
    if subset.empty:
        continue
    print(f"\n  ── {TIER_LABELS[tier]} ──")
    for fire, row in subset.iterrows():
        match_str   = f"{row['match_pct']:.1f}%" if pd.notna(row['match_pct']) else "N/A"
        matched_str = f"{int(row['n_matched']):,}" if pd.notna(row['n_matched']) else "N/A"
        ww_str      = f"{row['ww_med_m']:.1f} m"  if pd.notna(row['ww_med_m'])  else "N/A"
        print(f"    {fire:<28}  match={match_str:>6}  matched={matched_str:>6}"
              f"  ww_med={ww_str:>7}")
        print(f"    {'':28}  → {row['reason']}")

# ── Update batch_summary.csv ──────────────────────────────────────────
if (OUTPUT_ROOT / 'batch_summary.csv').exists():
    df_sum = pd.read_csv(OUTPUT_ROOT / 'batch_summary.csv')
    tier_map    = {r['fire']: r['tier']      for r in tier_records}
    matched_map = {r['fire']: r['n_matched'] for r in tier_records}
    df_sum['confidence_tier']    = df_sum['fire'].map(tier_map)
    df_sum['n_dins_matched_abs'] = df_sum['fire'].map(matched_map)
    df_sum.to_csv(OUTPUT_ROOT / 'batch_summary.csv', index=False)
    print(f"\n  batch_summary.csv updated with confidence_tier column.")

print(f"\n  Fires suitable for primary SSDD analysis (Tier A + B):")
primary = df_tier[df_tier['tier'].isin(['A', 'B'])]
print(f"  {len(primary)} / {len(df_tier)} fires")
for fire in primary.index:
    print(f"    {fire}")

---
## 13. Results Summary

In [ ]:
# Load and display the batch summary table
csv_path = OUTPUT_ROOT / 'batch_summary.csv'
if not csv_path.exists():
    print("No batch_summary.csv found — run Section 12 first.")
else:
    df = pd.read_csv(csv_path)
    success = df[df['status'] == 'success'].copy()
    failed  = df[df['status'] != 'success']

    print(f"Fires completed : {len(success)} / {len(df)}")
    if len(failed):
        print(f"Failed          : {list(failed['fire'])}")
    print()

    # Display key metrics for completed fires
    cols = ['fire','ignition_date','n_buildings_merged','dins_n_structures',
            'dins_match_pct','ww_median_m','ww_lt3m_pct','ww_lt15m_pct','elapsed_sec']
    cols = [c for c in cols if c in success.columns]
    display_df = success[cols].set_index('fire')

    # Rename for readability
    display_df.columns = [
        c.replace('n_buildings_merged','total_bldgs')
         .replace('dins_n_structures','dins_structs')
         .replace('dins_match_pct','dins_match_%')
         .replace('ww_median_m','ww_med_m')
         .replace('ww_lt3m_pct','<3m_%')
         .replace('ww_lt15m_pct','<15m_%')
         .replace('elapsed_sec','sec')
        for c in display_df.columns
    ]
    # Only apply gradient to columns that are actually present after renaming
    # (some may be absent if the batch is still in progress or DINS was unavailable)
    gradient_candidates = ['dins_match_%', '<3m_%']
    gradient_cols = [c for c in gradient_candidates if c in display_df.columns]
    try:
        if gradient_cols:
            display(display_df.style.background_gradient(
                subset=gradient_cols, cmap='YlOrRd'))
        else:
            display(display_df)
    except Exception:
        print(display_df.to_string())